# Parcial Final
# Electiva I
Fabian Andres Molina Arango


# Creacion de tablas


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.customers
USING DELTA
AS
SELECT *
FROM read_files(
    '/Volumes/workspace/electiva/mi_volumen_datos/customers.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => true
);
SELECT *
FROM workspace.electiva.customers
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.employee
USING DELTA
AS
SELECT *
FROM read_files(
    '/Volumes/workspace/electiva/mi_volumen_datos/employees.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => true
);
SELECT *
FROM workspace.electiva.employee
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.orders
USING DELTA
AS
SELECT *
FROM read_files(
    '/Volumes/workspace/electiva/mi_volumen_datos/orders.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => true
);
SELECT *
FROM workspace.electiva.orders
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.product_categories
USING DELTA
AS
SELECT *
FROM read_files(
    '/Volumes/workspace/electiva/mi_volumen_datos/productcategories.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => true
);
SELECT *
FROM workspace.electiva.    
product_categories;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.product_sub_categories
USING DELTA
AS
SELECT *
FROM read_files(
    '/Volumes/workspace/electiva/mi_volumen_datos/productsubcategories.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => true
);
SELECT *
FROM workspace.electiva.product_sub_categories;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.products
USING DELTA
AS
SELECT *
FROM read_files(
    '/Volumes/workspace/electiva/mi_volumen_datos/products.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => true
);
SELECT *
FROM workspace.electiva.products
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.vendor
USING DELTA
AS
SELECT *
FROM read_files(
    '/Volumes/workspace/electiva/mi_volumen_datos/vendors.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => true
);
SELECT *
FROM workspace.electiva.vendor
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.vendor_product
USING DELTA
AS
SELECT *
FROM read_files(
    '/Volumes/workspace/electiva/mi_volumen_datos/vendorproduct.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => true
);
SELECT *
FROM workspace.electiva.vendor_product
LIMIT 10;

# PUNTO 1

In [0]:
%sql
SELECT
    CustomerID AS customerid,
    FullName AS fullname,
CONCAT(
CAST(CustomerID AS STRING),
'-',
UPPER(SUBSTRING(FirstName, 1, 2)),
'-',
LOWER(SUBSTRING(LastName, 1, 2)),
'-AW'
    ) AS user_name
FROM workspace.electiva.customers;

# PUNTO 2


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.page_user (
    customerid INT,
    user_name STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO workspace.electiva.page_user
SELECT
CustomerID AS customerid,
CONCAT(
CAST(CustomerID AS STRING),
'-',
UPPER(SUBSTRING(FirstName, 1, 2)),
'-',
LOWER(SUBSTRING(LastName, 1, 2)),
'-AW'
    ) AS user_name
FROM workspace.electiva.customers;

In [0]:
%sql
SELECT *
FROM workspace.electiva.page_user;

# PUNTO 3

In [0]:
%sql
SELECT
    table_name,
    column_name,
    data_type
FROM workspace.information_schema.columns
WHERE table_schema = 'electiva'
AND table_name IN (
    'orders',
    'products',
    'product_sub_categories',
    'product_categories',
    'page_user'
)
ORDER BY table_name, ordinal_position;

In [0]:
%sql

CREATE OR REPLACE VIEW workspace.electiva.orders_august_2011 AS
WITH orders_base AS (
SELECT
        p.ProductName AS productname,
        p.ModelName AS modelname,
        pu.user_name AS user_name,
        o.SalesOrderID AS salesorderid,
        pc.CategoryName AS categoryname,
        o.OrderDate AS orderdate,
        o.LineTotal AS linetotal,
CASE
            WHEN pc.CategoryName ILIKE 'Bikes' THEN 0.10
            WHEN pc.CategoryName ILIKE 'Clothing' THEN 0.15
            WHEN pc.CategoryName ILIKE 'Components' THEN 0.08
            WHEN pc.CategoryName ILIKE 'Accessories' THEN 0.20
            ELSE 0
        END AS `descuento_mitad_año`
FROM workspace.electiva.orders o
INNER JOIN workspace.electiva.products p
        ON o.ProductID = p.ProductID
INNER JOIN workspace.electiva.product_sub_categories psc
        ON p.SubCategoryID = psc.SubCategoryID
INNER JOIN workspace.electiva.product_categories pc
        ON psc.CategoryID = pc.CategoryID
INNER JOIN workspace.electiva.page_user pu
        ON o.CustomerID = pu.customerid
WHERE o.OrderDate >= DATE '2011-08-01'
      AND o.OrderDate < DATE '2011-09-01'
)
SELECT
    productname,
    modelname,
    user_name,
    salesorderid,
    categoryname,
    orderdate,
    linetotal,
    `descuento_mitad_año`,
linetotal -
    (linetotal * `descuento_mitad_año`)
    AS total_descuentos_aplicados
FROM orders_base;

In [0]:
%sql
SELECT *
FROM workspace.electiva.orders_august_2011
LIMIT 20;

In [0]:
%sql
SELECT COUNT(*) AS total_registros,MIN(orderdate) AS fecha_inicial,MAX(orderdate) AS fecha_final
FROM workspace.electiva.orders_august_2011;

In [0]:
%sql
SELECT
    categoryname,
    `descuento_mitad_año`,
    COUNT(*) AS cantidad
FROM workspace.electiva.orders_august_2011
GROUP BY categoryname, `descuento_mitad_año`
ORDER BY categoryname;

# PUNTO 4

In [0]:
%sql
WITH ordenes_cliente AS ( 
    SELECT CustomerID,
        COUNT(DISTINCT SalesOrderID) AS amount_of_orders
 FROM workspace.electiva.orders
WHERE OrderDate >= DATE '2011-01-01'
      AND OrderDate < DATE '2012-01-01'
GROUP BY CustomerID
)
SELECT
amount_of_orders,
COUNT(*) AS cantidad_clientes
FROM ordenes_cliente
GROUP BY amount_of_orders
ORDER BY amount_of_orders;

In [0]:
%sql

CREATE OR REPLACE MATERIALIZED VIEW workspace.electiva.customer_class AS
WITH customer_orders AS (
SELECT
        CustomerID,
        COUNT(DISTINCT SalesOrderID) AS amount_of_orders
FROM workspace.electiva.orders
WHERE OrderDate >= DATE '2011-01-01'
      AND OrderDate < DATE '2012-01-01'
GROUP BY CustomerID
)
SELECT
co.amount_of_orders,
pu.user_name,
c.FullName AS customerfullname,
CASE
    WHEN co.amount_of_orders = 3 THEN 'Oro'
    WHEN co.amount_of_orders = 2 THEN 'Plata'
    WHEN co.amount_of_orders < 2 THEN 'Ocasional'
    END AS user_level,
c.CustomerID AS customerid
FROM customer_orders co
INNER JOIN workspace.electiva.customers c
    ON co.CustomerID = c.CustomerID
INNER JOIN workspace.electiva.page_user pu
    ON co.CustomerID = pu.customerid;

In [0]:
%sql

SELECT *
FROM workspace.electiva.customer_class
LIMIT 20;

In [0]:
%sql
SELECT
    user_level,
    COUNT(*) AS cantidad_clientes
FROM workspace.electiva.customer_class
GROUP BY user_level
ORDER BY user_level;